In [ ]:
using Pkg
Pkg.activate(".")
Pkg.develop(path="../../")
ENV["TAMBOSIM_PATH"] = realpath("../../")

In [ ]:
using Dierckx
using CairoMakie
using FHist
using HDF5
using Integrals
using Makie
using Tambo
using Unitful
include("../plotting_boilerplate.jl")

In [ ]:
filename = "../../resources/cross_section_tables/cross_sections.h5"
groupname = "CSMS_nutau"
cross_section = Tambo.CrossSection("$(filename):$(groupname)");

In [ ]:
es, zs, tot_xs, diff_xs, emin = h5open(filename) do file
    group = file[groupname]
    tot_xs = read(group["total_xs"]) * u"cm"^2
    diff_xs = read(group["differential_xs"]) * u"cm"^2
    es = read(group["energies"]) .* u"GeV"
    zs = read(group["zs"])
    emin = read(group["emin"]) * u"GeV"
    es, zs, tot_xs, diff_xs, emin
end

## Total cross sections

In [ ]:
fig = Figure()
ax = Axis(
    fig[1,1],
    xscale=log10,
    yscale=log10,
    xticks=(10 .^ (2:12), [L"10^{%$(x)}" for x in 2:12]),
    yticks=(10.0 .^ (-36:-31), [L"10^{%$(x)}" for x in -36:-31]),
)

lines!(
    ax,
    ustrip(es .|> u"GeV"),
    ustrip.(tot_xs .|> u"cm"^2),
    linewidth=5
)

lines!(
    ax,
    ustrip(es .|> u"GeV"),
    ustrip.(cross_section.(es) .|> u"cm"^2)
)

fig

## Differential cross sections
These have been computed in terms of

$$z = \frac{E-E_{\mathrm{min}}}{E_{\mathrm{in}} - E_{\mathrm{min}}},$$

where $E_{\mathrm{min}}$ is the lower bound of integration used in the cross section computation.

In [ ]:
zs_fine = 0:0.001:1

for idx in 1:50:size(diff_xs, 1)

    fig = Figure()
    ax1 = Axis(fig[1,1], height=300, limits=(0,1, 1e-38, 1e-29), xticklabelsvisible=false, yscale=log10)
    ax2 = Axis(fig[2,1], height=100, limits=(0, 1, 0.99, 1.01))
    
    ein = es[idx]
    eouts_fine = zs_fine .* (ein - emin) .+ emin
    eouts = zs .* (ein - emin) .+ emin
    
    lines!(ax1, zs, ustrip.(diff_xs[idx,:]), linewidth=3, label="Tabulated")
    lines!(ax1, zs_fine, ustrip.(cross_section.(Ref(ein), eouts_fine)), label="Spline")
    
    lines!(ax2, zs, ustrip.(cross_section.(Ref(ein), eouts)) ./ ustrip.(diff_xs[idx,:]))
    
    axislegend(ax1, position=:lt)
    
    resize_to_layout!(fig)

    display(fig)
end

## Sampling outgoing energies

In [ ]:
for idx in 1:50:size(diff_xs, 1)
    ein = es[idx]

    N = 100000
    nbin = 100
    
    outgoing_es = rand(cross_section, ein, N)
    new_zs = (outgoing_es .- emin) ./ (ein - emin)
    
    ys = ustrip.(diff_xs[idx, :])
    itp = Spline1D(zs, log.(ys))
    
    f(u, p) = exp(itp(u))
    domain = (minimum(zs), maximum(zs))
    prob = IntegralProblem(f, domain)
    sol = solve(prob, HCubatureJL(); reltol = 1e-10, abstol = 1e-10)
    
    h = Hist1D(ustrip.(new_zs); nbins=nbin)
    cents = (h.binedges[1][2:end] .+ h.binedges[1][1:end-1]) ./ 2
    
    fig = Figure()
    ax = Axis(fig[1,1], limits=(0,1, nothing, nothing), yscale=log10)
    
    stairs!(ax, cents, h.bincounts, step=:center, label="Samples", color=:crimson)
    lines!(ax, zs, ys ./ sol .* N / nbin, label="Normalize tabulated")
    
    axislegend(ax, position=:lb)
    
    display(fig)
    
end